# VBZ GTFS-Daten — Aufbereitung 2023–2025

Analyse und Filterung der GTFS-Fahrplandaten des ZVV-Kantons auf die relevanten VBZ-Tramlinien.

| Export | Inhalt | Verwendung |
| :--- | :--- | :--- |
| **`gtfs_tram_*.parquet`** | 16 VBZ-Tramlinien, 2023–2025 | Verspätungsanalyse & ML-Modell |
| **`gtfs_zurich_*.parquet`** | Gesamter Stadtverkehr Zürich | Kontext & Anschlussanalysen |

**Referenzjahr:** 2024 — vollständigste Datenlage, alle bekannten Linien vorhanden.

## Kontext: Strassenbahn Zürich

| | |
| :--- | :--- |
| Betreiber | Verkehrsbetriebe Zürich (VBZ) |
| Verbund | Zürcher Verkehrsverbund (ZVV) |
| Eröffnung | 1882 |
| Streckenlänge | 97 km |
| Spurweite | 1.000 mm (Meterspur) |
| Stromsystem | 600 Volt Gleichstrom, Oberleitung |
| Linien | 18 (Stand 2018) |
| Fahrgäste | 202,6 Mio. jährlich (2018) |

Quelle: [Wikipedia — Strassenbahn Zürich](https://de.wikipedia.org/wiki/Strassenbahn_Z%C3%BCrich)

---

## Wichtige Projektabgrenzung

> ⚠️ **Am 14. Dezember 2025 fand der grösste Fahrplanwechsel in der Geschichte der VBZ statt.**

Fünf Tramlinien erhielten neue Wegführungen, neue Linien wurden eingeführt (u.a. Tramnetz Süd).

**Für dieses Projekt bedeutet das:**
- Die Ist-Daten 2023–2025 spiegeln ausschliesslich das **alte Liniennetz**
- Ein direkter Vergleich mit dem aktuellen Netz (ab Dezember 2025) ist ohne Anpassung nicht möglich
- Bewusste Projektentscheidung: wir analysieren einen stabilen, vollständig dokumentierten Zeitraum

---

## Tramlinien im Analysezeitraum (altes Liniennetz bis Dezember 2025)

| Linie | Strecke |
| :--- | :--- |
| 2 | Schlieren Geissweid — Bahnhof Tiefenbrunnen |
| 3 | Albisrieden — Klusplatz |
| 4 | Bahnhof Altstetten Nord — Bahnhof Tiefenbrunnen |
| 5 | Laubegg — Zoo |
| 6 | Bahnhof Enge — Zoo |
| 7 | Wollishoferplatz — Bahnhof Stettbach |
| 8 | Klusplatz — Hardturm |
| 9 | Hirzenbach — Triemli |
| 10 | Hauptbahnhof — Zürich Flughafen |
| 11 | Rehalp — Auzelg |
| 12 | Bahnhof Stettbach — Zürich Flughafen |
| 13 | Albisgütli — Frankental |
| 14 | Triemli — Seebach |
| 15 | Bucheggplatz — Bahnhof Stadelhofen |
| 17 | Werdhölzli — Hauptbahnhof |
| S18 | Stadelhofen — Rehalp — Esslingen (Forchbahn AG) |

> **Linie 1 — historisch eingestellt:** Verkehrte bis 1954. Die Nummer ist reserviert — Tram Hohlstrasse nach 2030 geplant.
> **Linie 16 — nie vergeben:** Farbschema-Entscheidung zugunsten der Verwechslungsfreiheit zwischen Linie 15 (violett) und 17 (bordeaux).

---

## Export-Strategie

**Export A — `gtfs_tram`:** 16 konsistente VBZ-Tramlinien
- Ohne S18 / Forchbahn (kein VBZ-Betrieb, inkonsistentes Format über die Jahre)
- Basis für Verspätungsanalyse und Modell

**Export B — `gtfs_zurich`:** Gesamter Stadtverkehr Zürich
- Tram + Bus + S-Bahn + S18 (harmonisiert)
- Bounding Box Stadtgebiet: 47.30–47.45 N / 8.45–8.65 E
- Für spätere Kontext-Analysen und Anschlussketten

---

## Hinweis: Join mit IST-Daten (BPUIC → stop_id)

Die IST-Daten verwenden `BPUIC` als Haltestellen-ID — nicht `stop_id`. Der Join erfolgt über ein Mapping, das aus der `stop_url` der GTFS-Stops extrahiert wird:

```
stop_url enthält: input=BPUIC
1 BPUIC → mehrere stop_ids (Bahnsteigkanten ::0, ::50 etc.)
→ Für Join: ersten Eintrag pro BPUIC verwenden + Koordinaten mitnehmen
```

In [ ]:
import pandas as pd
from pathlib import Path

GTFS_BASE  = Path('../../data/raw/vbz/gtfs/')
OUTPUT_DIR = Path('../../data/interim/vbz/gtfs/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = ['2023_google_transit', '2024_google_transit', '2025_google_transit']

---

## 1 — Überblick Rohdaten

Alle verfügbaren GTFS-Dateien werden eingelesen und auf Zeilenzahl und Spalten geprüft.

> `stop_times.txt` (4–6 Mio. Zeilen) wird für dieses Projekt nicht benötigt — die IST-Daten enthalten bereits beide Zeitangaben: `ANKUNFTSZEIT` (Soll) und `AN_PROGNOSE` (Ist).

In [ ]:
# Dateigrössen und Zeilenzahlen pro Jahr
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for year in YEARS:
    gtfs_dir = GTFS_BASE / year
    print(f'\n{year}')
    for f in sorted(gtfs_dir.glob('*.txt')):
        df = pd.read_csv(f)
        print(f'  {f.name:<35} {len(df):>8,} Zeilen — {df.shape[1]} Spalten')

---

## 2 — Jahresvergleich

Prüfung ob sich Haltestellen und Routen zwischen den Jahren unterscheiden.

> **Hinweis:** 2023 verwendet ein anderes `stop_id`-Format als 2024/2025 → Vergleich über `stop_name`.

In [ ]:
# Haltestellen und Routen pro Jahr laden
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
stops_2023  = pd.read_csv(GTFS_BASE / '2023_google_transit/stops.txt')
stops_2024  = pd.read_csv(GTFS_BASE / '2024_google_transit/stops.txt')
stops_2025  = pd.read_csv(GTFS_BASE / '2025_google_transit/stops.txt')
routes_2023 = pd.read_csv(GTFS_BASE / '2023_google_transit/routes.txt')
routes_2024 = pd.read_csv(GTFS_BASE / '2024_google_transit/routes.txt')
routes_2025 = pd.read_csv(GTFS_BASE / '2025_google_transit/routes.txt')

# stop_id Format prüfen
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print('stop_id Formate:')
print('  2023:', stops_2023['stop_id'].head(2).tolist())
print('  2024:', stops_2024['stop_id'].head(2).tolist())
print('  2025:', stops_2025['stop_id'].head(2).tolist())

# Vergleich über stop_name (Format-unabhängig)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
names_2023 = set(stops_2023['stop_name'])
names_2024 = set(stops_2024['stop_name'])
names_2025 = set(stops_2025['stop_name'])

print('\n=== Haltestellen (via Name) ===')
print(f'Nur in 2023:      {len(names_2023 - names_2024 - names_2025)}')
print(f'Nur in 2024:      {len(names_2024 - names_2023 - names_2025)}')
print(f'Nur in 2025:      {len(names_2025 - names_2023 - names_2024)}')
print(f'In allen Jahren:  {len(names_2023 & names_2024 & names_2025)}')

# Routen-Vergleich
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
r_2023 = set(routes_2023['route_short_name'])
r_2024 = set(routes_2024['route_short_name'])
r_2025 = set(routes_2025['route_short_name'])

print('\n=== Routen ===')
print(f'Nur in 2023:  {r_2023 - r_2024 - r_2025}')
print(f'Nur in 2024:  {r_2024 - r_2023 - r_2025}')
print(f'Nur in 2025:  {r_2025 - r_2023 - r_2024}')
print(f'In allen:     {len(r_2023 & r_2024 & r_2025)} Routen')

### BPUIC-Mapping für Join mit IST-Daten

Die `stop_url` enthält die BPUIC-Nummer (Schlüssel der IST-Daten). Da 1 BPUIC mehrere `stop_id`s hat (Bahnsteigkanten `::0`, `::50` etc.), nehmen wir den ersten Eintrag pro BPUIC.

In [ ]:
# BPUIC aus stop_url extrahieren
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
stops_2024['bpuic'] = stops_2024['stop_url'].str.extract(r'input=(\d+)')

# Mapping: erster Eintrag pro BPUIC → stop_id + Koordinaten
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
bpuic_mapping = (
    stops_2024
    .dropna(subset=['bpuic'])
    .groupby('bpuic')
    .first()
    .reset_index()[['bpuic', 'stop_id', 'stop_name', 'stop_lat', 'stop_lon']]
)

print(f'Eindeutige BPUICs: {len(bpuic_mapping):,}')

# Beispiel: BPUIC aus IST-Daten prüfen
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
test_bpuic = '8590805'
match = stops_2024[stops_2024['bpuic'] == test_bpuic]
print(f'\nBPUIC {test_bpuic} → {len(match)} stop_ids (Bahnsteigkanten):')
print(match[['stop_id', 'stop_name']].to_string(index=False))

---

## 3 — Filterung auf Tram

GTFS enthält alle Verkehrsmittel des ZVV-Kantons. Filter auf `route_type = 0` (GTFS-Standard für Tram).

In [ ]:
# Tram-Routen pro Jahr
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for year, routes in [('2023', routes_2023), ('2024', routes_2024), ('2025', routes_2025)]:
    tram = routes[routes['route_type'] == 0]
    print(f'{year}: {len(tram)} Tram-Routen — {sorted(tram["route_short_name"].tolist())}')

# Schnittmenge: Linien in allen drei Jahren vorhanden
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
tram_all = (
    set(routes_2023[routes_2023['route_type'] == 0]['route_short_name']) &
    set(routes_2024[routes_2024['route_type'] == 0]['route_short_name']) &
    set(routes_2025[routes_2025['route_type'] == 0]['route_short_name'])
)
print(f'\nIn allen Jahren: {sorted(tram_all)}')

# S18 / Forchbahn: inkonsistentes Format über die Jahre
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print('\nForchbahn über die Jahre:')
for year, routes in [('2023', routes_2023), ('2024', routes_2024), ('2025', routes_2025)]:
    fb = routes[routes['route_short_name'].isin(['18', 'S18'])]
    if len(fb):
        print(f'  {year}: {fb[["route_short_name", "route_type"]].values.tolist()}')

---

## 4 — Export

Alle drei Jahre laden, `year`-Spalte hinzufügen, dann je nach Export filtern.

In [ ]:
# Alle drei Jahre laden
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
dfs = {}
for year in YEARS:
    y = year[:4]
    dfs[y] = {
        'routes': pd.read_csv(GTFS_BASE / year / 'routes.txt'),
        'stops':  pd.read_csv(GTFS_BASE / year / 'stops.txt'),
        'trips':  pd.read_csv(GTFS_BASE / year / 'trips.txt'),
        'shapes': pd.read_csv(GTFS_BASE / year / 'shapes.txt'),
    }
    for key in dfs[y]:
        dfs[y][key]['year'] = y
    print(f'{y} geladen')

In [ ]:
# Export A: gtfs_tram — 16 konsistente VBZ-Tramlinien
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
VBZ_TRAM_LINES = ['2','3','4','5','6','7','8','9','10','11','12','13','14','15','17','19']

LAT_MIN, LAT_MAX = 47.30, 47.45
LON_MIN, LON_MAX =  8.45,  8.65

tram_routes, tram_stops, tram_trips, tram_shapes = [], [], [], []

for y, data in dfs.items():
    r = data['routes']
    mask = (r['route_type'] == 0) & (r['route_short_name'].isin(VBZ_TRAM_LINES))
    routes_y = r[mask].copy()
    tram_routes.append(routes_y)

    trips_y = data['trips'][data['trips']['route_id'].isin(routes_y['route_id'])].copy()
    tram_trips.append(trips_y)
    tram_shapes.append(data['shapes'][data['shapes']['shape_id'].isin(trips_y['shape_id'])].copy())

    stops_y = data['stops'].copy()
    stops_y = stops_y[
        stops_y['stop_lat'].between(LAT_MIN, LAT_MAX) &
        stops_y['stop_lon'].between(LON_MIN, LON_MAX)
    ]
    tram_stops.append(stops_y)

df_tram_routes = pd.concat(tram_routes, ignore_index=True)
df_tram_trips  = pd.concat(tram_trips,  ignore_index=True)
df_tram_shapes = pd.concat(tram_shapes, ignore_index=True).drop_duplicates()
df_tram_stops  = pd.concat(tram_stops,  ignore_index=True).drop_duplicates(subset=['stop_id', 'year'])

df_tram_routes.to_parquet(OUTPUT_DIR / 'gtfs_tram_routes.parquet', index=False)
df_tram_trips.to_parquet(OUTPUT_DIR  / 'gtfs_tram_trips.parquet',  index=False)
df_tram_shapes.to_parquet(OUTPUT_DIR / 'gtfs_tram_shapes.parquet', index=False)
df_tram_stops.to_parquet(OUTPUT_DIR  / 'gtfs_tram_stops.parquet',  index=False)

print(f'gtfs_tram exportiert')
print(f'  Routen: {len(df_tram_routes)}')
print(f'  Trips:  {len(df_tram_trips):,}')
print(f'  Shapes: {len(df_tram_shapes):,}')
print(f'  Stops:  {len(df_tram_stops):,}')

In [ ]:
# Export B: gtfs_zurich — Gesamter Stadtverkehr Zürich
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
zurich_routes, zurich_stops, zurich_trips, zurich_shapes = [], [], [], []

for y, data in dfs.items():
    routes_y = data['routes'].copy()
    # S18 harmonisieren: Linie 18 (nur 2024) → S18
    routes_y.loc[routes_y['route_short_name'] == '18', 'route_short_name'] = 'S18'
    zurich_routes.append(routes_y)

    stops_y = data['stops'].copy()
    stops_y = stops_y[
        stops_y['stop_lat'].between(LAT_MIN, LAT_MAX) &
        stops_y['stop_lon'].between(LON_MIN, LON_MAX)
    ]
    zurich_stops.append(stops_y)

    trips_y = data['trips'][data['trips']['route_id'].isin(routes_y['route_id'])].copy()
    zurich_trips.append(trips_y)
    zurich_shapes.append(data['shapes'][data['shapes']['shape_id'].isin(trips_y['shape_id'])].copy())

df_zurich_routes = pd.concat(zurich_routes, ignore_index=True)
df_zurich_trips  = pd.concat(zurich_trips,  ignore_index=True)
df_zurich_shapes = pd.concat(zurich_shapes, ignore_index=True).drop_duplicates()
df_zurich_stops  = pd.concat(zurich_stops,  ignore_index=True).drop_duplicates(subset=['stop_id', 'year'])

df_zurich_routes.to_parquet(OUTPUT_DIR / 'gtfs_zurich_routes.parquet', index=False)
df_zurich_trips.to_parquet(OUTPUT_DIR  / 'gtfs_zurich_trips.parquet',  index=False)
df_zurich_shapes.to_parquet(OUTPUT_DIR / 'gtfs_zurich_shapes.parquet', index=False)
df_zurich_stops.to_parquet(OUTPUT_DIR  / 'gtfs_zurich_stops.parquet',  index=False)

print(f'gtfs_zurich exportiert')
print(f'  Routen: {len(df_zurich_routes)}')
print(f'  Trips:  {len(df_zurich_trips):,}')
print(f'  Shapes: {len(df_zurich_shapes):,}')
print(f'  Stops:  {len(df_zurich_stops):,}')

In [ ]:
# Übersicht exportierte Parquet-Dateien
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f'\n{OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.glob('*.parquet')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<35} {size_mb:.1f} MB')

---

## 5 — Ergebnis

| File | Inhalt | Grösse |
| :--- | :--- | :--- |
| `gtfs_tram_routes.parquet` | 16 VBZ-Tramlinien × 3 Jahre | ~0.0 MB |
| `gtfs_tram_trips.parquet` | Fahrten der Tramlinien | ~1.8 MB |
| `gtfs_tram_shapes.parquet` | Linienverläufe (Koordinaten) | ~4.7 MB |
| `gtfs_tram_stops.parquet` | Haltestellen im Stadtgebiet | ~0.5 MB |
| `gtfs_zurich_routes.parquet` | Alle Zürich-Routen (Bus, Tram, S-Bahn) | ~0.0 MB |
| `gtfs_zurich_trips.parquet` | Alle Fahrten Stadtgebiet | ~7.8 MB |
| `gtfs_zurich_shapes.parquet` | Alle Linienverläufe Stadtgebiet | ~26.2 MB |
| `gtfs_zurich_stops.parquet` | Alle Haltestellen Stadtgebiet | ~0.2 MB |

**Bounding Box Zürich:** 47.30–47.45 N / 8.45–8.65 E  
**Gesamt:** ~41 MB — handlich für alle weiteren Analysen.